# Feature engineering new ideas

In [1]:
import pandas as pd
import numpy as np


In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test-full.csv")

In [3]:
def create_features_plus(df):
    df = df.copy()

    df["Distance_To_Hydrology"] = np.sqrt(
        df["Horizontal_Distance_To_Hydrology"] ** 2
        + df["Vertical_Distance_To_Hydrology"] ** 2
    )

    df["Aspect_sin"] = np.sin(df["Aspect"] * np.pi / 180)
    df["Aspect_cos"] = np.cos(df["Aspect"] * np.pi / 180)
    
    df.drop(columns=["Aspect"], inplace=True)


    df["Slope_x_Aspect_sin"] = df["Slope"] * df["Aspect_sin"]
    df["Slope_x_Aspect_cos"] = df["Slope"] * df["Aspect_cos"]

    stony_cols = [f"Soil_Type{i}" for i in [1, 2, 6, 9, 12, 18, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 39, 40] if f"Soil_Type{i}" in df.columns]
    df["Total_Stony_Soil"] = df[stony_cols].sum(axis=1)

    df["Hillshade_Mean"] = (df["Hillshade_9am"] + df["Hillshade_Noon"] + df["Hillshade_3pm"]) / 3

    df["Diff_Elev_Hydrology"] = df["Elevation"] - df["Vertical_Distance_To_Hydrology"]

    df["Mean_Distance_Amenities"] = (df["Horizontal_Distance_To_Roadways"] + 
                                 df["Horizontal_Distance_To_Fire_Points"]) / 2

    df["Hillshade_AM_PM_Diff"] = df["Hillshade_9am"] - df["Hillshade_3pm"]

    df["Elevation_x_Slope"] = df["Elevation"] * df["Slope"]


    if "Soil_Type15" in df.columns:
        df.drop(columns=["Soil_Type15"], inplace=True)
        
    return df

In [4]:
train_fe = create_features_plus(train)
test_fe = create_features_plus(test)

In [8]:
train_fe.to_csv("data/train_new_processed.csv", index=False)
test_fe.to_csv("data/test_new_processed.csv", index=False)